# 18. Анализ ошибок (Kaggle)

Где и почему ошибается выбранная модель (гибрид `hyb_blend`) по сравнению с LightGBM и эталоном (`src/error_analysis.py`):
откуда берётся ошибка, худшие дни и станции, события у площадок, сбои по оповещениям MTA.
Заодно проверяется правило `hyb_ev`: в часы событий у площадок брать LightGBM вместо гибрида.

**Input:** датасет `vkr-transit-data`, вывод ноутбука 17 (`hybrid-resid`, в нём все прогнозы),
датасет с оповещениями MTA (папка `data/raw/alerts` с файлами `alerts_*.parquet`).
Accelerator — None, Internet — On. Время ~30–40 мин. Результат — `results_csv.zip` в Output.

In [ ]:
import os, glob, shutil, subprocess, time
REPO = "/kaggle/working/vkr-transit"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kinggl3/vkr-transit", REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
lab = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ridership_wide.parquet", recursive=True)]
assert lab, "Не найден ridership_wide.parquet: подключите датасет vkr-transit-data (Add Input)"
os.environ["VKR_LAB"] = lab[0]
os.environ["VKR_RESULTS"] = R = "/kaggle/working/results"
os.environ["PYTHONWARNINGS"] = "ignore"
os.makedirs(f"{R}/preds", exist_ok=True)
print("Данные:", lab[0], "| граф:", os.path.exists(os.path.join(os.path.dirname(lab[0]), "graph")),
      "| events.csv:", os.path.exists(os.path.join(lab[0], "events.csv")))
print("Подключено в /kaggle/input:")
for p in sorted(glob.glob("/kaggle/input/*/*")):
    print("  ", p)

## 1. Прогнозы и оповещения

In [ ]:
import pandas as pd
for p in sorted(glob.glob("/kaggle/input/**/results/preds/*.npz", recursive=True)):
    dst = f"{R}/preds/{os.path.basename(p)}"
    if not os.path.exists(dst):
        shutil.copy(p, dst)
logs = glob.glob("/kaggle/input/**/results/runs.csv", recursive=True)
if logs:
    pd.concat([pd.read_csv(p) for p in logs]).drop_duplicates().to_csv(f"{R}/runs.csv", index=False)
need = [f"{m}__{s}.npz" for m in ("hyb_blend", "lgbm", "lgbm_final", "norm_level") for s in ("val", "test")]
missing = [n for n in need if not os.path.exists(f"{R}/preds/{n}")]
assert not missing, f"Нет прогнозов {missing}: подключите вывод ноутбука 17"
al = sorted(glob.glob("/kaggle/input/**/alerts_*.parquet", recursive=True))
ALERTS = os.path.dirname(al[0]) if al else ""
ALERT_ARG = f"--alerts {ALERTS}" if ALERTS else ""
print("Оповещения:", ALERTS or "не подключены — раздел про сбои будет пропущен", f"({len(al)} файлов)")

## 2. Анализ ошибок

In [ ]:
!cd {REPO} && python src/error_analysis.py --split val {ALERT_ARG}

In [ ]:
!cd {REPO} && python src/error_analysis.py --split test {ALERT_ARG}

## 3. Правило hyb_ev в общей оценке

In [ ]:
!cd {REPO} && python src/evaluate.py --split val --ref norm_level

In [ ]:
!cd {REPO} && python src/evaluate.py --split test --ref norm_level

In [ ]:
PAIRS = "hyb_ev:hyb_blend hyb_ev:lgbm_final hyb_blend:lgbm"
!cd {REPO} && python src/report.py --split val --pairs {PAIRS}
!cd {REPO} && python src/report.py --split test --pairs {PAIRS}

## 4. Архив

In [ ]:
import zipfile
with zipfile.ZipFile("/kaggle/working/results_csv.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{R}/*.csv"):
        z.write(p, os.path.basename(p))
print(sorted(os.path.basename(p) for p in glob.glob(f"{R}/*.csv")))